# Contract Overlap — Algorithm Evaluation

## Objective
Evaluate the accuracy of the existing N2IA Contract Overlap implementation.

We will assess:
1. Lexical similarity — identification of contracts with related terminology.
2. Semantic similarity — identification of contracts with meaningfully related services.
3. Final screening — identification of contract relationships warranting further review.

**Approach:** Import and execute the same Python functions used by the production MCP Server, compare their predictions against independently reviewed contract pairs, and calculate precision, recall, and F1.

**Initial test:** FY2025 Cybersecurity contracts across GSA and DHS.

## 1. Install Evaluation Dependencies

Install the Python libraries required by the existing MCP Server's semantic analysis.

The evaluation notebook uses its own Python environment, separate from the deployed MCP Server App.

In [0]:
%pip install -U sentence-transformers
%pip install --upgrade databricks-mcp databricks-sdk "mcp>=1.24,<2" "networkx>=3.0,<4"
%pip install -U databricks-openai
dbutils.library.restartPython()

## 2. Load Existing MCP Server Code

Add the Git repository's `src` directory to the Python import path.

Import the existing contract retrieval and comparison functions. No similarity algorithms are reimplemented in this notebook.

In [0]:
import os
import sys

print("Current directory:", os.getcwd())
print("Python path:", sys.path[:3])
print("Files:", os.listdir(os.getcwd()))

In [0]:
from pathlib import Path
import sys

# Notebook is inside mcp_server_n2ia/eval
project_root = Path.cwd().parent.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from mcp_server_n2ia.services import overlap
from mcp_server_n2ia.services.overlap import Award

print("Successfully imported:", overlap.__file__)

## 3. Retrieve FY2025 Contract Data

Retrieve Cybersecurity contract awards from GSA and DHS using the MCP Server's existing retrieval function.

The function uses the configured USAspending retrieval/cache mechanism.

Fiscal year boundaries:
- Start: October 1, 2024
- End: September 30, 2025

This step establishes the contract population used for evaluation.

In [0]:
from datetime import date

# Same retrieval parameters as our production FY2025 test
retrieved = await overlap.retrieve_contracts(
    start_date=date(2024, 10, 1),
    end_date=date(2025, 9, 30),
    agencies=[
        "General Services Administration",
        "Department of Homeland Security",
    ],
    keyword="Cybersecurity",
    pages_per_agency=2,
    page_size=100,
)

if "error" in retrieved:
    raise RuntimeError(retrieved["error"])

awards = [
    Award.model_validate(item)
    for item in retrieved["items"]
]

print("Retrieved:", len(awards))

In [0]:
import importlib
import inspect
from datetime import date

from mcp_server_n2ia import config
from mcp_server_n2ia.services import overlap

importlib.reload(config)
importlib.reload(overlap)

source = inspect.getsource(overlap.group_overlap_candidates)

print("Updated lexical screening:",
      "lexical_supported" in source)

print("Updated semantic screening:",
      "semantic_supported" in source)

print("SME service function:",
      "cybersecurity_sme_support" in overlap.SERVICE_FUNCTION_RULES)

result = overlap.compare_awards(
    awards=awards,
    cross_agency_only=True,
    max_examples_per_category=10,
    analysis_start=date(2024, 10, 1),
    analysis_end=date(2025, 9, 30),
)

print("Thresholds:", result["thresholds"])
print("Contracts analyzed:", result["analyzed_awards"])
print("Eligible pairs:", result["eligible_pairs_compared"])
print("Preliminary candidates:", result["ranked_candidate_count"])
print("Qualified groups:", result["group_count"])

## 4. Execute Lexical and Semantic Comparison

Run the existing `compare_awards()` function with the same parameters used by the production MCP tool.

The analysis includes:
- Lexical matching threshold: 0.30
- Semantic matching threshold: 0.70
- Cross-agency comparisons only
- Performance-period comparisons restricted to FY2025

This is a reproducibility test, not yet an accuracy evaluation.

In [0]:
from datetime import date
from time import perf_counter

start = perf_counter()

result = overlap.compare_awards(
    awards=awards,
    min_lexical_similarity=0.2,
    min_semantic_similarity=0.55,
    cross_agency_only=True,
    max_examples_per_category=10,
    analysis_start=date(2024, 10, 1),
    analysis_end=date(2025, 9, 30),
)

print("Execution time:", round(perf_counter() - start, 2), "seconds")
print("Contracts analyzed:", result.get("analyzed_awards"))
print("Eligible pairs compared:", result.get("eligible_pairs_compared"))
print("Preliminary candidates:", result.get("ranked_candidate_count"))
print("Qualified groups:", result.get("group_count"))
print("Comparison categories:", result.get("counts"))

In [0]:

import inspect

print("ACTIVE THRESHOLDS:")
print(result.get("thresholds"))

print("\nLEXICAL IMPLEMENTATION:")
source = inspect.getsource(overlap.compare_awards)

print("Uses TF-IDF:", "TfidfVectorizer" in source)
print("Uses 2-word minimum:", "len(shared) >= 2" in source)

print("\nGROUPING RESULTS:")
print("Qualified relationships:",
      result.get("grouping_relationship_count"))
print("Qualified groups:", len(result.get("groups", [])))


In [0]:

print("Preliminary candidates:", result["ranked_candidate_count"])
print("Total rejected:", result["grouping_screened_out_relationships"])
print("Rejected for service-function mismatch:",
      result["grouping_screened_out_function_mismatch"])
print("Rejected before service-function screening:",
      result["grouping_screened_out_relationships"]
      - result["grouping_screened_out_function_mismatch"])
print("Qualified relationships:", result["grouping_relationship_count"])


In [0]:

for group in result["groups"]:
    contracts = {c["source_award_key"]: c for c in group["contracts"]}

    print("=" * 90)
    print("GROUP:", group["group_id"])
    print("Contracts:", group["contract_count"])
    print("Relationships:", group["relationship_count"])

    for relationship in group["relationships"]:
        a = contracts[relationship["award_key_a"]]
        b = contracts[relationship["award_key_b"]]

        print("\nContract A:", a["award_id"])
        print("Description A:", a["description"])
        print("Contract B:", b["award_id"])
        print("Description B:", b["description"])
        print("Lexical:", relationship["lexical_similarity"])
        print("Semantic:", relationship["semantic_similarity"])
        print("Shared terms:", relationship["shared_terms"])
        print("Shared functions:", relationship["shared_service_functions"])



## 28. Examine Final Screening Decisions by Contract Pair

For each preliminary candidate, inspect:
- Semantic similarity and threshold result.
- PSC/NAICS classification alignment.
- Number of shared non-generic terms.
- Shared service functions.

Identify which screening conditions prevent qualification.


In [0]:

import pandas as pd

rows = []

for pair in result["ranked_candidates"]:
    specific = sorted(
        set(pair["shared_terms"]) - overlap.GENERIC_GROUP_TERMS
    )

    left_functions = set(
        overlap.classify_service_functions(pair["left"]["description"])
    )
    right_functions = set(
        overlap.classify_service_functions(pair["right"]["description"])
    )

    common_functions = sorted(left_functions & right_functions)

    rows.append({
        "Contract A": pair["left"]["award_id"],
        "Contract B": pair["right"]["award_id"],
        "Semantic": pair["semantic_similarity"],
        "Lexical": pair["lexical_similarity"],
        "Semantic Pass": pair["semantic_similarity"] >= 0.55,
        "PSC/NAICS Match": pair["matching_psc"] or pair["matching_naics"],
        "Specific Terms": ", ".join(specific),
        "Specific Count": len(specific),
        "Shared Functions": ", ".join(common_functions),
    })

display(pd.DataFrame(rows))


## 5. Validate Against Production Results

Compare notebook results against the previously observed FY2025 MCP Server results.

Expected baseline:
- 117 retrieved awards
- 111 analyzed awards
- 1,113 eligible pairs
- 5 preliminary candidates
- 0 qualified groups

Matching results confirm that the notebook is executing the production comparison logic consistently.

This does not yet demonstrate that the identified relationships are accurate.

In [0]:
baseline = {
    "retrieved": 117,
    "analyzed": 111,
    "eligible_pairs": 1113,
    "preliminary_candidates": 5,
    "qualified_groups": 0,
}

actual = {
    "retrieved": len(awards),
    "analyzed": comparison.get("analyzed_awards"),
    "eligible_pairs": comparison.get("eligible_pairs_compared"),
    "preliminary_candidates": comparison.get("ranked_candidate_count"),
    "qualified_groups": comparison.get("group_count"),
}

for metric, expected in baseline.items():
    observed = actual[metric]
    status = "PASS" if observed == expected else "DIFFERENT"
    print(f"{metric}: {observed} (expected {expected}) — {status}")


## 6. Inspect Lexical and Semantic Matching Results

Examine the contract pairs identified by the production comparison algorithm.

This step helps us understand:
- Which contracts were identified as potentially related.
- Their lexical and semantic similarity scores.
- Whether their performance periods overlap.
- Why the final screening accepted or rejected the relationships.

These are algorithm-generated findings, not human-validated matches.


In [0]:

import pandas as pd

rows = []

for pair in comparison.get("ranked_candidates", []):
    left = pair.get("left") or {}
    right = pair.get("right") or {}

    rows.append({
        "Award A": left.get("award_id"),
        "Award B": right.get("award_id"),
        "Agency A": left.get("awarding_agency"),
        "Agency B": right.get("awarding_agency"),
        "Description A": left.get("description"),
        "Description B": right.get("description"),
        "Lexical Score": pair.get("lexical_similarity"),
        "Semantic Score": pair.get("semantic_similarity"),
        "Overlap Days": pair.get("overlap_days"),
        "Match Method": pair.get("match_method"),
        "Review Reason": pair.get("review_reason"),
    })

candidates_df = pd.DataFrame(rows)
display(candidates_df)

print("Candidate pairs displayed:", len(candidates_df))
print("Total preliminary candidates:",
      comparison.get("ranked_candidate_count"))



## 7. Examine Lexical vs. Semantic Matching

Inspect the existing comparison results to determine how many contract
pairs matched lexically, semantically, or through both methods.

This helps establish whether the Hugging Face semantic model is
identifying relationships that lexical matching misses.

A semantic match is a model-generated prediction, not proof that
two contracts are genuinely related.


In [0]:

counts = comparison.get("counts", {})

print("Comparison categories:")
for category, count in counts.items():
    print(f"  {category}: {count}")

print("\nSemantic matching examples:")

examples = comparison.get("examples", {})

for category in ("both", "semantic_only"):
    pairs = examples.get(category, [])
    print(f"\n{category}: {len(pairs)} returned examples")

    for pair in pairs[:5]:
        left = pair.get("left") or {}
        right = pair.get("right") or {}

        print(
            left.get("award_id"),
            "<->",
            right.get("award_id"),
            "| Semantic:",
            pair.get("semantic_similarity"),
            "| Lexical:",
            pair.get("lexical_similarity"),
        )



## 8. Examine Semantic Similarity Scores

Inspect the semantic similarity scores assigned to the preliminary
lexical matches.

The objective is to determine whether these candidates are close
to the current semantic threshold of 0.70.

This is an initial diagnostic. It does not represent the score
distribution across all eligible contract pairs.


In [0]:

scores = candidates_df[
    ["Award A", "Award B", "Lexical Score", "Semantic Score"]
].copy()

scores = scores.sort_values(
    "Semantic Score",
    ascending=False
)

display(scores)

print("Maximum semantic score:", scores["Semantic Score"].max())
print("Minimum semantic score:", scores["Semantic Score"].min())
print("Current semantic threshold:", 0.70)


## 9. Evaluate Semantic Similarity Across All Eligible Pairs

The current analysis returned five lexical matches and no semantic matches
at the 0.70 threshold.

This section will examine semantic scores across all eligible contract
pairs, including pairs rejected by lexical matching.

Objectives:
- Identify the highest semantic similarity scores.
- Examine semantic relationships missed by lexical matching.
- Determine whether the current semantic threshold is appropriate.
- Prepare representative pairs for independent human review.

This evaluation will use the existing MCP Server similarity implementation,
without changing production thresholds.


## 10. Evaluate All Eligible Contract Pairs

Calculate lexical and semantic similarity for every eligible
cross-agency contract pair using the existing MCP Server implementation.

This evaluation:
- Reuses the production Hugging Face embedding function.
- Uses the same tokenization and lexical similarity formula.
- Applies the same FY2025 performance-period boundaries.
- Includes pairs that production filtering normally excludes.
- Retains both scores for subsequent accuracy evaluation.

The production MCP Server is not modified.


In [0]:

from itertools import combinations
from math import sqrt
from datetime import date
import pandas as pd

from mcp_server_n2ia.services.semantic_similarity import calculate_embeddings

analysis_start = date(2024, 10, 1)
analysis_end = date(2025, 9, 30)

# Same deduplication and eligibility rules as compare_awards().
unique = {award.source_award_key: award for award in awards}

valid = [
    award for award in unique.values()
    if overlap.interval(award) is not None
    and len(overlap.tokens(award.description)) >= 3
]

# Existing production semantic model and tokenization.
embeddings = calculate_embeddings(
    [award.description for award in valid]
)

token_sets = [overlap.tokens(a.description) for a in valid]
intervals = [overlap.interval(a) for a in valid]

rows = []

for i, j in combinations(range(len(valid)), 2):
    left, right = valid[i], valid[j]

    # Same cross-agency eligibility rules.
    if not left.awarding_agency or not right.awarding_agency:
        continue
    if left.awarding_agency == right.awarding_agency:
        continue

    left_start, left_end = intervals[i]
    right_start, right_end = intervals[j]

    start = max(left_start, right_start, analysis_start)
    end = min(left_end, right_end, analysis_end)

    if start > end:
        continue

    shared = token_sets[i] & token_sets[j]

    lexical_score = len(shared) / sqrt(
        len(token_sets[i]) * len(token_sets[j])
    )

    semantic_score = float(embeddings[i] @ embeddings[j])

    lexical_match = (
        len(shared) >= 3 and lexical_score >= 0.30
    )

    semantic_match = semantic_score >= 0.70

    category = (
        "both" if lexical_match and semantic_match
        else "lexical_only" if lexical_match
        else "semantic_only" if semantic_match
        else "neither"
    )

    rows.append({
        "award_key_a": left.source_award_key,
        "award_key_b": right.source_award_key,
        "award_id_a": left.award_id,
        "award_id_b": right.award_id,
        "agency_a": left.awarding_agency,
        "agency_b": right.awarding_agency,
        "description_a": left.description,
        "description_b": right.description,
        "lexical_score": round(lexical_score, 4),
        "semantic_score": round(semantic_score, 4),
        "shared_word_count": len(shared),
        "overlap_days": (end - start).days + 1,
        "matching_psc": bool(
            left.psc and right.psc and left.psc == right.psc
        ),
        "matching_naics": bool(
            left.naics and right.naics and left.naics == right.naics
        ),
        "category": category,
    })

pairs_df = pd.DataFrame(rows)

print("Eligible pairs:", len(pairs_df))
print("\nMatching categories:")
print(pairs_df["category"].value_counts())

assert len(pairs_df) == comparison["eligible_pairs_compared"]
print("\nPair-count verification: PASS")



## 11. Examine the Strongest Semantic Relationships

Rank all eligible pairs by semantic similarity, regardless of whether
they passed the current threshold.

This helps us identify:
- The highest semantic similarity across the dataset.
- Whether strong semantic relationships were missed by lexical matching.
- Whether the 0.70 threshold may be overly restrictive.

Similarity scores are algorithm predictions, not human-reviewed truth.


In [0]:

top_semantic = pairs_df.sort_values(
    "semantic_score",
    ascending=False
).head(20)

display(top_semantic[[
    "award_id_a",
    "award_id_b",
    "description_a",
    "description_b",
    "lexical_score",
    "semantic_score",
    "shared_word_count",
    "overlap_days",
    "category",
]])

print("Maximum semantic score:",
      pairs_df["semantic_score"].max())

print("Median semantic score:",
      pairs_df["semantic_score"].median())

print("Pairs >= 0.70:",
      (pairs_df["semantic_score"] >= 0.70).sum())

print("Pairs >= 0.60:",
      (pairs_df["semantic_score"] >= 0.60).sum())

print("Pairs >= 0.50:",
      (pairs_df["semantic_score"] >= 0.50).sum())



## 12. Review Top Semantic Matches

Examine contract pairs with the highest semantic similarity scores.

Objectives:
- Identify potentially related contracts missed by lexical matching.
- Compare contract descriptions and agency missions.
- Determine whether high semantic similarity corresponds to meaningful service similarity.
- Prepare independent human judgments for subsequent precision and recall evaluation.

Do not classify pairs as related solely because their semantic scores are high.


In [0]:

review_df = (
    pairs_df.sort_values("semantic_score", ascending=False)
    .head(20)
    .copy()
)

review_df["human_similar_services"] = None
review_df["human_review_notes"] = ""

display(review_df[[
    "award_id_a",
    "award_id_b",
    "description_a",
    "description_b",
    "lexical_score",
    "semantic_score",
    "category",
    "human_similar_services",
    "human_review_notes",
]])



## 13. Semantic Similarity Threshold Sensitivity

Evaluate how different semantic similarity thresholds affect
the number of contract pairs identified.

Objectives:
- Understand how sensitive match counts are to the threshold.
- Compare alternative thresholds against the current 0.70 setting.
- Identify a reasonable threshold range for further investigation.

This analysis measures matching behavior, not whether matches are correct.


In [0]:

import pandas as pd

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

results = []

for threshold in thresholds:
    matches = pairs_df[
        pairs_df["semantic_score"] >= threshold
    ]

    results.append({
        "Threshold": threshold,
        "Matching Pairs": len(matches),
        "Percentage of Eligible Pairs": round(
            len(matches) / len(pairs_df) * 100, 2
        ),
    })

threshold_df = pd.DataFrame(results)

display(threshold_df)



## 14. Semantic Discovery Beyond Lexical Matching

Examine contract pairs with semantic similarity of at least 0.55.

Determine how many:
- Also satisfy lexical matching requirements.
- Were missed by lexical matching.
- Would be newly identified using a 0.55 semantic threshold.

This evaluates incremental discovery, not confirmed accuracy.


In [0]:

semantic_candidates = pairs_df[
    pairs_df["semantic_score"] >= 0.5
].copy()

semantic_candidates["lexical_match"] = (
    (semantic_candidates["lexical_score"] >= 0.30)
    & (semantic_candidates["shared_word_count"] >= 3)
)

print("Semantic candidates (>=0.5):", len(semantic_candidates))
print("Also matched lexically:",
      semantic_candidates["lexical_match"].sum())
print("Missed by lexical matching:",
      (~semantic_candidates["lexical_match"]).sum())

display(
    semantic_candidates.sort_values(
        "semantic_score", ascending=False
    )[[
        "award_id_a",
        "award_id_b",
        "lexical_score",
        "semantic_score",
        "shared_word_count",
        "lexical_match",
        "description_a",
        "description_b",
    ]]
)



## 17. Lexical Similarity — Detailed Explanation

Examine the highest-scoring contract pairs to understand how lexical
similarity is calculated.

For each pair, display:
- Contract A and Contract B descriptions.
- Informative words extracted from each description.
- Shared words contributing to the similarity score.
- Number of shared words and total informative words.
- Calculated lexical score and final match decision.

This analysis uses the existing production tokenization logic,
including the current STOPWORDS list.

No production code is modified.


In [0]:

from math import sqrt
import pandas as pd

# Examine the 10 highest lexical similarity scores.
top_lexical = pairs_df.sort_values(
    "lexical_score", ascending=False
).head(10)

for _, pair in top_lexical.iterrows():
    tokens_a = overlap.tokens(pair["description_a"])
    tokens_b = overlap.tokens(pair["description_b"])

    shared = tokens_a & tokens_b
    only_a = tokens_a - tokens_b
    only_b = tokens_b - tokens_a

    score = len(shared) / sqrt(
        len(tokens_a) * len(tokens_b)
    )

    lexical_match = len(shared) >= 3 and score >= 0.30

    print("=" * 90)
    print(f"CONTRACT A: {pair['award_id_a']}")
    print(f"Description: {pair['description_a']}\n")

    print(f"CONTRACT B: {pair['award_id_b']}")
    print(f"Description: {pair['description_b']}\n")

    print("SHARED INFORMATIVE WORDS:")
    print(sorted(shared))

    print("\nONLY IN CONTRACT A:")
    print(sorted(only_a))

    print("\nONLY IN CONTRACT B:")
    print(sorted(only_b))

    print("\nLEXICAL CALCULATION:")
    print(f"Shared words: {len(shared)}")
    print(f"Informative words in A: {len(tokens_a)}")
    print(f"Informative words in B: {len(tokens_b)}")
    print(
        f"Score = {len(shared)} / "
        f"sqrt({len(tokens_a)} x {len(tokens_b)}) "
        f"= {score:.4f}"
    )

    print(f"Semantic score: {pair['semantic_score']:.4f}")
    print(f"Lexical match: {'YES' if lexical_match else 'NO'}")
    print()



## 18. Evaluate Improved Lexical Similarity Using TF-IDF

Test TF-IDF cosine similarity as an alternative to the existing lexical matching algorithm.

The current algorithm:
- Assigns equal importance to every informative word.
- Requires at least three shared informative words.
- Uses a minimum lexical similarity score of 0.30.

The proposed TF-IDF approach:
- Gives greater importance to distinctive words.
- Reduces the influence of frequently occurring terminology.
- Uses the existing production tokenization and stopword rules.
- Does not impose the three-shared-word requirement.

Both methods are evaluated against the same FY2025 eligible contract pairs.

This experiment does not modify the production MCP Server.


In [0]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Use the same valid contract population as production.
descriptions = [a.description or "" for a in valid]

# Reuse production tokenization and stopwords.
vectorizer = TfidfVectorizer(
    analyzer=lambda text: sorted(overlap.tokens(text)),
    lowercase=False,
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
)

tfidf_matrix = vectorizer.fit_transform(descriptions)

# Map each contract to its TF-IDF vector.
award_index = {
    a.source_award_key: i
    for i, a in enumerate(valid)
}

# Calculate TF-IDF cosine similarity for the same
# eligible pairs already evaluated in pairs_df.
tfidf_scores = []

for _, pair in pairs_df.iterrows():
    i = award_index[pair["award_key_a"]]
    j = award_index[pair["award_key_b"]]

    score = tfidf_matrix[i].multiply(
        tfidf_matrix[j]
    ).sum()

    tfidf_scores.append(float(score))

pairs_df["tfidf_score"] = np.round(tfidf_scores, 4)

# Preserve the production lexical match decision.
pairs_df["current_lexical_match"] = (
    (pairs_df["lexical_score"] >= 0.30)
    & (pairs_df["shared_word_count"] >= 3)
)

# Compare scores on the same contract pairs.
comparison_df = pairs_df[[
    "award_id_a",
    "award_id_b",
    "description_a",
    "description_b",
    "shared_word_count",
    "lexical_score",
    "tfidf_score",
    "semantic_score",
    "current_lexical_match"
]].copy()

display(
    comparison_df.sort_values(
        "tfidf_score", ascending=False
    ).head(20)
)

print("Pairs evaluated:", len(comparison_df))
print("Existing lexical matches:",
      comparison_df["current_lexical_match"].sum())
print("Maximum TF-IDF score:",
      comparison_df["tfidf_score"].max())
print("Median TF-IDF score:",
      comparison_df["tfidf_score"].median())



## 19. Compare TF-IDF Against Known Contract Examples

Compare the existing lexical score, TF-IDF score, and semantic score
for contract pairs previously examined.

Determine whether TF-IDF gives higher relative rankings to
service-specific relationships than to generic vocabulary matches.

This evaluates ranking quality, not validated matching accuracy.


In [0]:

# Contract pairs previously reviewed.
review_pairs = [
    ("47QFSA23F0087", "70RCSJ24FR0000053"),  # SME support
    ("47QFLA24C0008", "70RCSJ24FR0000027"),  # Architecture / engineering
    ("47QFPA21F0100", "70Z04425FCAD20001"), # RMF services
    ("47QFCA20F0049", "70LGLY22FSSB00121"), # Enterprise IT / FLETC
    ("47QFSA25F0001", "70T04023F7672N015")  # OT training / authentication
]

results = []

for a, b in review_pairs:
    match = comparison_df[
        (
            (comparison_df["award_id_a"] == a)
            & (comparison_df["award_id_b"] == b)
        )
        | (
            (comparison_df["award_id_a"] == b)
            & (comparison_df["award_id_b"] == a)
        )
    ]

    if not match.empty:
        results.append(match.iloc[0])

review_df = pd.DataFrame(results)

display(review_df[[
    "award_id_a",
    "award_id_b",
    "lexical_score",
    "tfidf_score",
    "semantic_score",
    "shared_word_count",
    "current_lexical_match"
]].sort_values("tfidf_score", ascending=False))



## 20. Compare Top 15 Lexical vs. TF-IDF Matches

Compare the highest-ranked contract pairs using:

1. Existing lexical similarity (equal-weight words).
2. TF-IDF cosine similarity (frequency-weighted words).

For each pair, examine:
- Its ranking under both methods.
- The words contributing to the match.
- Whether it appears in both top-15 lists.
- The original contract descriptions.

This evaluates ranking quality independently of the production
three-shared-word matching requirement.

No production code is modified.


In [0]:

import pandas as pd

# Rank all eligible pairs using both methods.
ranked = pairs_df.copy()

ranked["lexical_rank"] = (
    ranked["lexical_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranked["tfidf_rank"] = (
    ranked["tfidf_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Select top 15 from each method.
top_lexical = ranked.sort_values(
    "lexical_score", ascending=False
).head(15)

top_tfidf = ranked.sort_values(
    "tfidf_score", ascending=False
).head(15)

# Build a combined view using the original award keys.
top_keys = set(
    zip(top_lexical["award_key_a"], top_lexical["award_key_b"])
) | set(
    zip(top_tfidf["award_key_a"], top_tfidf["award_key_b"])
)

comparison_rows = []

for _, pair in ranked.iterrows():
    key = (pair["award_key_a"], pair["award_key_b"])

    if key not in top_keys:
        continue

    words_a = overlap.tokens(pair["description_a"])
    words_b = overlap.tokens(pair["description_b"])

    comparison_rows.append({
        "Contract A": pair["award_id_a"],
        "Contract B": pair["award_id_b"],
        "Lexical Rank": pair["lexical_rank"],
        "TF-IDF Rank": pair["tfidf_rank"],
        "Lexical Score": pair["lexical_score"],
        "TF-IDF Score": pair["tfidf_score"],
        "Semantic Score": pair["semantic_score"],
        "Shared Words": ", ".join(sorted(words_a & words_b)),
        "In Lexical Top 15": key in set(zip(
            top_lexical["award_key_a"],
            top_lexical["award_key_b"]
        )),
        "In TF-IDF Top 15": key in set(zip(
            top_tfidf["award_key_a"],
            top_tfidf["award_key_b"]
        )),
        "Description A": pair["description_a"],
        "Description B": pair["description_b"],
    })

ranking_comparison_df = pd.DataFrame(comparison_rows)

print("Top 15 lexical pairs:", len(top_lexical))
print("Top 15 TF-IDF pairs:", len(top_tfidf))
print(
    "Pairs appearing in both top-15 lists:",
    (
        ranking_comparison_df["In Lexical Top 15"]
        & ranking_comparison_df["In TF-IDF Top 15"]
    ).sum()
)

display(
    ranking_comparison_df.sort_values(
        ["TF-IDF Rank", "Lexical Rank"]
    )
)


In [0]:

import pandas as pd

configs = {
    "Current lexical": None,
    "Hybrid A (0.20, 2 words)": (0.20, 2),
    "Hybrid B (0.15, 2 words)": (0.15, 2),
    "Hybrid C (0.20, 3 words)": (0.20, 3),
}

results = []

current = (
    (pairs_df["lexical_score"] >= 0.30)
    & (pairs_df["shared_word_count"] >= 3)
)

for name, config in configs.items():
    if config is None:
        matches = current
    else:
        threshold, min_words = config
        matches = (
            (pairs_df["tfidf_score"] >= threshold)
            & (pairs_df["shared_word_count"] >= min_words)
        )

    results.append({
        "Configuration": name,
        "Matching Pairs": int(matches.sum()),
        "New vs Current": int((matches & ~current).sum()),
        "Current Matches Retained": int((matches & current).sum()),
        "Current Matches Lost": int((~matches & current).sum()),
    })

hybrid_results_df = pd.DataFrame(results)

display(hybrid_results_df)



## 22. Review Hybrid B's New and Lost Contract Matches

Compare Hybrid B (TF-IDF >= 0.15, minimum 2 shared words)
against the current production lexical algorithm.

Review:
- Seven newly identified contract pairs.
- One existing lexical match rejected by Hybrid B.
- Contract descriptions and shared informative words.

Determine whether the changes improve the relevance of identified
contract relationships before modifying production code.


In [0]:

current = (
    (pairs_df["lexical_score"] >= 0.30)
    & (pairs_df["shared_word_count"] >= 3)
)

hybrid_b = (
    (pairs_df["tfidf_score"] >= 0.15)
    & (pairs_df["shared_word_count"] >= 2)
)

review = pairs_df[
    (hybrid_b & ~current) | (current & ~hybrid_b)
].copy()

review["Change"] = [
    "NEW - Hybrid B" if hybrid_b.loc[idx]
    else "LOST - Current lexical"
    for idx in review.index
]

review["Shared Words"] = review.apply(
    lambda row: ", ".join(sorted(
        overlap.tokens(row["description_a"])
        & overlap.tokens(row["description_b"])
    )),
    axis=1
)

display(
    review[[
        "Change",
        "award_id_a",
        "award_id_b",
        "lexical_score",
        "tfidf_score",
        "semantic_score",
        "shared_word_count",
        "Shared Words",
        "description_a",
        "description_b"
    ]].sort_values(
        ["Change", "tfidf_score"],
        ascending=[False, False]
    )
)
